# Лабораторная работа №3
## Сравнение реализаций нейронных сетей: Keras vs PyTorch vs собственная реализация

**Выполнил:** Привалихин Дмитрий Сергеевич, ИСИб-23-1

**Вариант:** tanh, α=3, T0=0

In [1]:
# Импорты библиотек
import os
import random
import numpy as np
import copy
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter

# Настройки
IMG_SIZE = 32  # Размер картинки 32x32 пикселя
ALPHABET = "йклмнопрсту"  # 11 букв для распознавания
NUM_SAMPLES_PER_LETTER = 1000  # По 1000 примеров на букву
DATASET_DIR = "dataset_images"  # Папка для картинок

# Вариант: tanh с коэффициентом 3
ALPHA = 3  # Делает функцию активации круче
T0 = 0  # Смещение

**Что здесь:**
- Загружаем библиотеки для работы с изображениями и массивами
- Задаём размер 32×32, 11 букв, по 1000 примеров = 11000 картинок
- ALPHA=3 делает tanh круче, сеть быстрее учится

In [2]:
# Генерация датасета
def get_local_fonts():
    # Ищем шрифты в папке проекта, если нет - берём системные
    font_paths = [f for f in os.listdir('.') if f.endswith('.ttf')]
    if not font_paths:
        sys_path = "C:\\Windows\\Fonts\\"
        common_fonts = ['arial.ttf', 'times.ttf', 'verdana.ttf']
        font_paths = [os.path.join(sys_path, f) for f in common_fonts if os.path.exists(os.path.join(sys_path, f))]
    if not font_paths:
        raise RuntimeError("Шрифты .ttf не найдены!")
    return font_paths

def generate_char_image(char, font_path, save_path=None):
    # Создаём картинку с буквой + аугментация
    canvas_size = IMG_SIZE * 3  # Холст больше для качества
    bg_color = random.randint(235, 255)  # Светлый фон
    img = Image.new('L', (canvas_size, canvas_size), bg_color)
    draw = ImageDraw.Draw(img)

    # Случайный размер шрифта
    font_size = random.randint(int(IMG_SIZE * 0.9), int(IMG_SIZE * 1.3))
    font = ImageFont.truetype(font_path, font_size)

    # Центрируем букву
    bbox = draw.textbbox((0, 0), char, font=font)
    w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (canvas_size - w) / 2 - bbox[0]
    y = (canvas_size - h) / 2 - bbox[1]
    draw.text((x, y), char, font=font, fill=0)

    # Поворот на случайный угол
    img = img.rotate(random.uniform(-15, 15), resample=Image.BICUBIC, fillcolor=bg_color)
    # Размытие с вероятностью 30%
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 0.8)))

    # Вырезаем центр 32x32 со смещением
    center = canvas_size // 2
    s = IMG_SIZE // 2
    jx, jy = random.randint(-2, 2), random.randint(-2, 2)
    img = img.crop((center - s + jx, center - s + jy, center + s + jx, center + s + jy))

    if save_path:
        img.save(save_path)

    # Инвертируем и нормализуем к [0, 1]
    img = ImageOps.invert(img)
    arr = np.array(img).astype(np.float32) / 255.0
    # Добавляем шум с вероятностью 40%
    if random.random() < 0.4:
        arr += np.random.normal(0, 0.01, arr.shape)

    # Бинаризуем
    arr = (np.clip(arr, 0, 1) > 0.4).astype(np.float32)
    return arr.flatten()

def generate_dataset():
    fonts = get_local_fonts()
    all_images = []
    all_labels = []

    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATASET_DIR)
        print(f"Создана директория: {DATASET_DIR}")

    print("Генерация данных...")
    for label, char in enumerate(ALPHABET):
        char_dir = os.path.join(DATASET_DIR, char)
        if not os.path.exists(char_dir):
            os.makedirs(char_dir)

        for i in range(NUM_SAMPLES_PER_LETTER):
            try:
                f = random.choice(fonts)
                c = char.upper() if random.random() < 0.3 else char  # 30% заглавных
                img_name = f"img_{i}.png"
                save_path = os.path.join(char_dir, img_name)
                img_vector = generate_char_image(c, f, save_path=save_path)
                all_images.append(img_vector)
                all_labels.append(label)
            except Exception as e:
                continue

    print(f"Генерация завершена. Всего {len(all_images)} изображений.")
    return np.array(all_images), np.array(all_labels)

# Запуск генерации
X, y = generate_dataset()

Создана директория: dataset_images
Генерация данных...
Генерация завершена. Всего 11000 изображений.


**Аугментация (чтобы сеть лучше работала):**
- Разные шрифты
- Поворот ±15°
- Размытие (30%)
- Шум (40%)
- Смещение при кадрировании

**Итог:** 11000 картинок (11 букв × 1000)

In [3]:
# Собственная реализация нейронной сети
class Network:
    # Полносвязная сеть с tanh(α*(x-T0))
    
    def __init__(self, sizes):
        # sizes = [1024, 128, 64, 11] - размеры слоёв
        self.num_layers = len(sizes)
        self.sizes = sizes
        # Смещения
        self.biases = [np.random.randn(y, 1) for y in sizes[1:]]
        # Веса (инициализация He)
        self.weights = [np.random.randn(y, x) * np.sqrt(2 / x) for x, y in zip(sizes[:-1], sizes[1:])]
        self.history = []

    def tanh(self, z):
        # tanh с коэффициентом 3
        return np.tanh(ALPHA * (z - T0))

    def tanh_prime(self, z):
        # Производная tanh
        tanh_val = self.tanh(z)
        return ALPHA * (1 - tanh_val ** 2)

    def softmax(self, z):
        # Softmax для классификации
        exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
        return exp_z / np.sum(exp_z, axis=0, keepdims=True)

    def feedforward(self, a):
        # Прямой проход
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, a) + b
            # Последний слой - softmax, остальные - tanh
            a = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
        return a

    def SGD(self, training_data, epochs, mini_batch_size, eta,
            test_data=None, early_stopping=True, patience=5, monitor='val_loss'):
        # Обучение SGD с мини-батчами
        n = len(training_data)
        if test_data:
            n_test = len(test_data)

        best_weights = copy.deepcopy(self.weights)
        best_biases = copy.deepcopy(self.biases)
        best_metric = float('inf') if monitor == 'val_loss' else 0
        wait = 0

        for epoch in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k + mini_batch_size] for k in range(0, n, mini_batch_size)]

            for mini_batch in mini_batches:
                x_batch = np.hstack([x for x, y in mini_batch])
                y_batch = np.hstack([y for x, y in mini_batch])
                self.update_mini_batch(x_batch, y_batch, eta)

            train_acc = self.evaluate([(x, np.argmax(y)) for x, y in training_data]) / n
            train_loss = self.compute_loss(training_data)

            val_acc = val_loss = None
            if test_data:
                val_acc = self.evaluate(test_data) / n_test
                val_loss = self.compute_loss(test_data, is_test=True)

            log = {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc
            }
            self.history.append(log)

            print_str = f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f}"
            if test_data:
                print_str += f" - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
            print(print_str)

            if early_stopping and test_data:
                current_metric = val_loss if monitor == 'val_loss' else val_acc
                is_improved = (current_metric < best_metric) if monitor == 'val_loss' else (current_metric > best_metric)

                if is_improved:
                    best_metric = current_metric
                    best_weights = copy.deepcopy(self.weights)
                    best_biases = copy.deepcopy(self.biases)
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"Early stopping at epoch {epoch+1}. Best {monitor}: {best_metric:.4f}")
                        break

        self.weights = best_weights
        self.biases = best_biases

    def update_mini_batch(self, x_batch, y_batch, eta):
        # Обновление весов
        nabla_b, nabla_w = self.backprop(x_batch, y_batch)
        batch_size = x_batch.shape[1]
        self.weights = [w - (eta / batch_size) * nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta / batch_size) * nb for b, nb in zip(self.biases, nabla_b)]

    def backprop(self, x_batch, y_batch):
        # Обратное распространение
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]

        activation = x_batch
        activations = [x_batch]
        zs = []

        # Прямой проход
        for i, (b, w) in enumerate(zip(self.biases, self.weights)):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = self.softmax(z) if i == self.num_layers - 2 else self.tanh(z)
            activations.append(activation)

        # Обратный проход
        delta = activations[-1] - y_batch
        nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-1] = np.dot(delta, activations[-2].T)

        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = self.tanh_prime(z)
            delta = np.dot(self.weights[-l + 1].T, delta) * sp
            nabla_b[-l] = np.sum(delta, axis=1, keepdims=True)
            nabla_w[-l] = np.dot(delta, activations[-l - 1].T)

        return nabla_b, nabla_w

    def evaluate(self, test_data):
        # Точность на тесте
        test_results = [(np.argmax(self.feedforward(x)), y) for (x, y) in test_data]
        return sum(int(x == y) for (x, y) in test_results)

    def compute_loss(self, data, is_test=False):
        # Кросс-энтропия
        total_loss = 0.0
        for x, y in data:
            if is_test:
                y = self.vectorize_label(y)
            output = self.feedforward(x)
            loss = -np.sum(y * np.log(output + 1e-8))
            total_loss += loss
        return total_loss / len(data)

    def vectorize_label(self, label):
        # One-hot вектор
        e = np.zeros((self.sizes[-1], 1))
        e[label] = 1.0
        return e

**Собственная сеть:**
- Слои: 1024 → 128 → 64 → 11
- Активация: tanh(3x) в скрытых, softmax на выходе
- SGD с мини-батчами
- Early stopping

In [4]:
# Подготовка данных
num_classes = len(ALPHABET)
test_size = 0.2
n_samples = len(X)
indices = np.random.permutation(n_samples)
test_count = int(test_size * n_samples)

test_indices = indices[:test_count]
train_indices = indices[test_count:]

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

# One-hot кодирование
def to_categorical(labels, num_classes):
    result = np.zeros((len(labels), num_classes))
    for i, label in enumerate(labels):
        result[i, label] = 1.0
    return result

y_train_cat = to_categorical(y_train, num_classes)

# Для своей сети - столбцы
X_train = [x.reshape(-1, 1) for x in X_train]
X_test = [x.reshape(-1, 1) for x in X_test]
y_train = [y.reshape(-1, 1) for y in y_train_cat]
training_data = list(zip(X_train, y_train))
test_data = list(zip(X_test, y_test))

input_size = IMG_SIZE * IMG_SIZE

print(f"Обучающая: {len(training_data)}, Тестовая: {len(test_data)}")

Обучающая: 8800, Тестовая: 2200


**Данные:**
- 80% обучение (8800), 20% тест (2200)
- One-hot для своей сети
- Индексы для Keras/PyTorch

In [6]:
# Обучение собственной модели
print("\n=== ОБУЧЕНИЕ МОЕЙ СЕТИ ===")
my_net = Network([input_size, 128, 64, num_classes])
my_net.SGD(training_data, epochs=150, mini_batch_size=32, eta=0.1,
           test_data=test_data, early_stopping=True, patience=30, monitor='val_loss')

print(f"\nЛучшая val_acc: {max([h['val_acc'] for h in my_net.history if h['val_acc']]):.4f}")


=== ОБУЧЕНИЕ МОЕЙ СЕТИ ===
Epoch 1/150 - loss: 0.2091 - acc: 0.9415 - val_loss: 0.2352 - val_acc: 0.9291
Epoch 2/150 - loss: 0.0963 - acc: 0.9730 - val_loss: 0.1269 - val_acc: 0.9632
Epoch 3/150 - loss: 0.0487 - acc: 0.9885 - val_loss: 0.0974 - val_acc: 0.9677
Epoch 4/150 - loss: 0.0505 - acc: 0.9853 - val_loss: 0.0921 - val_acc: 0.9695
Epoch 5/150 - loss: 0.0170 - acc: 0.9973 - val_loss: 0.0642 - val_acc: 0.9773
Epoch 6/150 - loss: 0.0165 - acc: 0.9974 - val_loss: 0.0527 - val_acc: 0.9845
Epoch 7/150 - loss: 0.0093 - acc: 0.9989 - val_loss: 0.0472 - val_acc: 0.9845
Epoch 8/150 - loss: 0.0034 - acc: 1.0000 - val_loss: 0.0348 - val_acc: 0.9891
Epoch 9/150 - loss: 0.0027 - acc: 1.0000 - val_loss: 0.0339 - val_acc: 0.9891
Epoch 10/150 - loss: 0.0023 - acc: 1.0000 - val_loss: 0.0339 - val_acc: 0.9891
Epoch 11/150 - loss: 0.0021 - acc: 1.0000 - val_loss: 0.0338 - val_acc: 0.9900
Epoch 12/150 - loss: 0.0018 - acc: 1.0000 - val_loss: 0.0340 - val_acc: 0.9905
Epoch 13/150 - loss: 0.0017 - acc

**Обучение своей сети:**
- 150 эпох, мини-батч 32, lr=0.1
- Early stopping с patience=30
- Сохраняем лучшие веса

In [ ]:
# Keras реализация
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Кастомная активация tanh(3x)
def custom_tanh(x):
    return tf.nn.tanh(3.0 * x)

# Регистрируем для сохранения
keras.utils.get_custom_objects()['custom_tanh'] = custom_tanh

# Данные для Keras (32x32x1)
X_train_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_train]).astype('float32')
X_test_k = np.array([x.reshape(IMG_SIZE, IMG_SIZE, 1) for x in X_test]).astype('float32')

# Метки индексами
y_train_idx = np.array([np.argmax(y) for y in y_train_cat]).astype('int64')
y_test_idx = y_test.astype('int64')

# Модель
keras_model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Flatten(),
    layers.Dense(256),
    layers.Activation(custom_tanh),
    layers.Dense(128),
    layers.Activation(custom_tanh),
    layers.Dense(num_classes, activation='softmax')
])

keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== Keras модель ===")
keras_model.summary()


=== Keras модель ===


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 11)             │           715 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,171 (547.54 KB)

 Trainable params: 140,171 (547.54 KB)

 Non-trainable params: 0 (0.00 B)

**Keras:**
- Flatten → Dense(128) + tanh(3x) → Dense(64) + tanh(3x) → Dense(11) + softmax
- Adam lr=0.0005
- sparse_categorical_crossentropy

In [ ]:
# Обучение Keras
print("\n=== ОБУЧЕНИЕ KERAS ===")
keras_history = keras_model.fit(
    X_train_k, y_train_idx,
    epochs=150,
    batch_size=32,
    validation_data=(X_test_k, y_test_idx),
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
    ],
    verbose=1
)

print(f"\nЛучшая accuracy: {max(keras_history.history['val_accuracy']):.4f}")


=== ОБУЧЕНИЕ KERAS ===
Epoch 1/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7962 - loss: 0.7325 - val_accuracy: 0.9359 - val_loss: 0.2953
Epoch 2/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9692 - loss: 0.1659 - val_accuracy: 0.9591 - val_loss: 0.1644
Epoch 3/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9917 - loss: 0.0713 - val_accuracy: 0.9723 - val_loss: 0.1103
Epoch 4/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9960 - loss: 0.0381 - val_accuracy: 0.9741 - val_loss: 0.0937
Epoch 5/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9991 - loss: 0.0222 - val_accuracy: 0.9800 - val_loss: 0.0711
Epoch 6/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9999 - loss: 0.0128 - val_accuracy: 0.9832 - val_loss: 0.0597
Epoch 7/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9999 - loss: 0.0090 - val_accuracy: 0.9827 - val_loss: 0.0588
Epoch 8/150
275/275 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 1.0000 -

## Подбор лучшего гиперпараметров


In [11]:
"""
Простой подбор гиперпараметров для Keras
"""
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Параметры
IMG_SIZE = 32
num_classes = 11
ALPHABET = "йклмнопрсту"

# Загрузка данных (предполагается, что они уже есть)
# X_train_k, y_train_idx, X_val_k, y_val_idx - должны быть подготовлены

print("="*50)
print("ПОДБОР ГИПЕРПАРАМЕТРОВ KERAS")
print("="*50)

best_acc = 0
best_params = {}

# Перебор параметров
for alpha in [1, 2, 3]:
    for lr in [0.0005, 0.001]:
        for units in [(128, 64), (256, 128)]:
            print(f"\nalpha={alpha}, lr={lr}, units={units}")
            
            def custom_tanh(x):
                return tf.nn.tanh(alpha * x)
            
            keras.utils.get_custom_objects()['custom_tanh'] = custom_tanh
            
            model = keras.Sequential([
                layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
                layers.Flatten(),
                layers.Dense(units[0]),
                layers.Activation(custom_tanh),
                layers.Dense(units[1]),
                layers.Activation(custom_tanh),
                layers.Dense(num_classes, activation='softmax')
            ])
            
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            
            history = model.fit(
                X_train_k, y_train_idx,
                epochs=30,
                batch_size=32,
                validation_data=(X_test_k, y_test_idx),
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
                verbose=0
            )
            
            val_acc = max(history.history['val_accuracy'])
            print(f"  -> val_acc = {val_acc:.4f}")
            
            if val_acc > best_acc:
                best_acc = val_acc
                best_params = {'alpha': alpha, 'lr': lr, 'units': units, 'val_acc': val_acc}

print("\n" + "="*50)
print("ЛУЧШИЕ ПАРАМЕТРЫ:")
print(f"  alpha  = {best_params['alpha']}")
print(f"  lr     = {best_params['lr']}")
print(f"  units  = {best_params['units']}")
print(f"  acc    = {best_params['val_acc']:.4f}")
print("="*50)


ПОДБОР ГИПЕРПАРАМЕТРОВ KERAS

alpha=1, lr=0.0005, units=(128, 64)
  -> val_acc = 0.9914

alpha=1, lr=0.0005, units=(256, 128)
  -> val_acc = 0.9932

alpha=1, lr=0.001, units=(128, 64)
  -> val_acc = 0.9950

alpha=1, lr=0.001, units=(256, 128)
  -> val_acc = 0.9968

alpha=2, lr=0.0005, units=(128, 64)
  -> val_acc = 0.9936

alpha=2, lr=0.0005, units=(256, 128)
  -> val_acc = 0.9941

alpha=2, lr=0.001, units=(128, 64)
  -> val_acc = 0.9927

alpha=2, lr=0.001, units=(256, 128)
  -> val_acc = 0.9950

alpha=3, lr=0.0005, units=(128, 64)
  -> val_acc = 0.9909

alpha=3, lr=0.0005, units=(256, 128)
  -> val_acc = 0.9945

alpha=3, lr=0.001, units=(128, 64)
  -> val_acc = 0.9936

alpha=3, lr=0.001, units=(256, 128)
  -> val_acc = 0.9973

ЛУЧШИЕ ПАРАМЕТРЫ:
  alpha  = 3
  lr     = 0.001
  units  = (256, 128)
  acc    = 0.9973


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import copy
from itertools import product

# --- Настройки устройства ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# --- Конфигурация гиперпараметров ---
alphas = [1, 2, 3]
lrs = [0.0005, 0.001]
layer_configs = [(128, 64), (256, 128)]

# Генерируем все комбинации
combinations = list(product(alphas, lrs, layer_configs))

# --- Динамическая архитектура модели ---
class PyTorchNet(nn.Module):
    def __init__(self, input_size, units, num_classes, alpha):
        super().__init__()
        self.alpha = alpha
        u1, u2 = units
        
        self.fc1 = nn.Linear(input_size, u1)
        self.fc2 = nn.Linear(u1, u2)
        self.fc3 = nn.Linear(u2, num_classes)
        
        # Инициализация Xavier
        for m in [self.fc1, self.fc2, self.fc3]:
            nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = torch.tanh(self.alpha * self.fc1(x))
        x = torch.tanh(self.alpha * self.fc2(x))
        x = self.fc3(x)
        return x

X_train_pt = torch.FloatTensor(np.array([x.flatten() for x in X_train])).to(device)
y_train_pt = torch.LongTensor(np.array([np.argmax(y) for y in y_train_cat])).to(device)
X_test_pt = torch.FloatTensor(np.array([x.flatten() for x in X_test])).to(device)
y_test_pt = torch.LongTensor(y_test.astype(np.int64)).to(device)

train_dataset = TensorDataset(X_train_pt, y_train_pt)

# --- Цикл подбора ---
pytorch_results = []

print("\n" + "="*70)
print(f"{'ПОДБОР ГИПЕРПАРАМЕТРОВ PYTORCH':^70}")
print("="*70)

for alpha_val, lr_val, units_val in combinations:
    print(f"\n>>> Тест: alpha={alpha_val}, lr={lr_val}, units={units_val}")
    
    # Инициализация модели и инструментов
    model = PyTorchNet(input_size, units_val, num_classes, alpha_val).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr_val)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    best_val_acc = 0
    best_val_loss = float('inf')
    patience = 10
    wait = 0
    
    # Обучение (50 эпох с Early Stopping)
    for epoch in range(50):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_test_pt)
            v_loss = criterion(val_outputs, y_test_pt).item()
            _, predicted = torch.max(val_outputs, 1)
            v_acc = (predicted == y_test_pt).sum().item() / len(y_test_pt)
            
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_val_loss = v_loss
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    pytorch_results.append({
        'alpha': alpha_val,
        'lr': lr_val,
        'units': str(units_val),
        'acc': best_val_acc,
        'loss': best_val_loss
    })
    print(f"Результат: Acc={best_val_acc:.4f}, Loss={best_val_loss:.4f}")

print("\n" + "="*70)
print(f"{'ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ':^70}")
print("-" * 70)
print(f"{'Alpha':<7} | {'LR':<8} | {'Units':<15} | {'Val Acc':<10} | {'Val Loss':<10}")
print("-" * 70)

for res in sorted(pytorch_results, key=lambda x: x['acc'], reverse=True):
    print(f"{res['alpha']:<7} | {res['lr']:<8} | {res['units']:<15} | {res['acc']:<10.4f} | {res['loss']:<10.4f}")

best_final = max(pytorch_results, key=lambda x: x['acc'])
print("-" * 70)
print(f"ЛУЧШИЙ ВАРИАНТ: Alpha={best_final['alpha']}, LR={best_final['lr']}, Units={best_final['units']}")
print(f"Точность: {best_final['acc']:.4f}")
print("="*70)

Используемое устройство: cpu

                    ПОДБОР ГИПЕРПАРАМЕТРОВ PYTORCH                    

>>> Тест: alpha=1, lr=0.0005, units=(128, 64)
Результат: Acc=0.9905, Loss=0.0343

>>> Тест: alpha=1, lr=0.0005, units=(256, 128)
Результат: Acc=0.9914, Loss=0.0292

>>> Тест: alpha=1, lr=0.001, units=(128, 64)
Результат: Acc=0.9959, Loss=0.0191

>>> Тест: alpha=1, lr=0.001, units=(256, 128)
Результат: Acc=0.9968, Loss=0.0176

>>> Тест: alpha=2, lr=0.0005, units=(128, 64)
Результат: Acc=0.9941, Loss=0.0264

>>> Тест: alpha=2, lr=0.0005, units=(256, 128)
Результат: Acc=0.9955, Loss=0.0220

>>> Тест: alpha=2, lr=0.001, units=(128, 64)
Результат: Acc=0.9936, Loss=0.0228

>>> Тест: alpha=2, lr=0.001, units=(256, 128)
Результат: Acc=0.9955, Loss=0.0195

>>> Тест: alpha=3, lr=0.0005, units=(128, 64)
Результат: Acc=0.9932, Loss=0.0314

>>> Тест: alpha=3, lr=0.0005, units=(256, 128)
Результат: Acc=0.9923, Loss=0.0273

>>> Тест: alpha=3, lr=0.001, units=(128, 64)
Результат: Acc=0.9936, Loss=0.02

In [13]:
# PyTorch реализация
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

class PyTorchNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        # tanh(3.0 * x)
        x = torch.tanh(3.0 * self.fc1(x))
        x = torch.tanh(3.0 * self.fc2(x))
        x = self.fc3(x)
        return x

# Данные
X_train_arr = np.array([x.flatten() for x in X_train])
X_test_arr = np.array([x.flatten() for x in X_test])
y_train_indices = np.array([np.argmax(y) for y in y_train_cat], dtype=np.int64)
y_test_indices = y_test.astype(np.int64)

X_train_pt = torch.FloatTensor(X_train_arr).to(device)
y_train_pt = torch.LongTensor(y_train_indices).to(device)
X_test_pt = torch.FloatTensor(X_test_arr).to(device)
y_test_pt = torch.LongTensor(y_test_indices).to(device)

train_dataset = TensorDataset(X_train_pt, y_train_pt)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

pt_model = PyTorchNet(input_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=0.00001)

print("\n=== PyTorch модель ===")
print(pt_model)

Устройство: cpu

=== PyTorch модель ===
PyTorchNet(
  (fc1): Linear(in_features=1024, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=11, bias=True)
)


**PyTorch:**
- Linear(1024→128) + tanh(3x) → Linear(128→64) + tanh(3x) → Linear(64→11)
- CrossEntropyLoss (включает softmax)
- Adam lr=0.001 + scheduler

In [14]:
# Обучение PyTorch
print("\n=== ОБУЧЕНИЕ PYTORCH ===")
best_val_loss = float('inf')
best_model_state = None
patience = 30
wait = 0

for epoch in range(150):
    pt_model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = pt_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    pt_model.eval()
    with torch.no_grad():
        val_outputs = pt_model(X_test_pt)
        val_loss = criterion(val_outputs, y_test_pt).item()
        _, val_predicted = torch.max(val_outputs.data, 1)
        val_acc = (val_predicted == y_test_pt).sum().item() / len(y_test_pt)
    
    scheduler.step(val_loss)
    
    train_acc = correct / total
    print(f"Epoch {epoch+1}/150 - loss: {total_loss/len(train_loader):.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss - 0.0001:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(pt_model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best val_loss: {best_val_loss:.4f}")
            break

if best_model_state:
    pt_model.load_state_dict(best_model_state)

print(f"\nЛучший val_loss: {best_val_loss:.4f}")


=== ОБУЧЕНИЕ PYTORCH ===
Epoch 1/150 - loss: 0.4426 - acc: 0.8675 - val_loss: 0.1190 - val_acc: 0.9682
Epoch 2/150 - loss: 0.0652 - acc: 0.9849 - val_loss: 0.0847 - val_acc: 0.9750
Epoch 3/150 - loss: 0.0265 - acc: 0.9947 - val_loss: 0.0680 - val_acc: 0.9786
Epoch 4/150 - loss: 0.0218 - acc: 0.9948 - val_loss: 0.0564 - val_acc: 0.9832
Epoch 5/150 - loss: 0.0190 - acc: 0.9964 - val_loss: 0.0525 - val_acc: 0.9827
Epoch 6/150 - loss: 0.0134 - acc: 0.9958 - val_loss: 0.0412 - val_acc: 0.9877
Epoch 7/150 - loss: 0.0041 - acc: 0.9994 - val_loss: 0.0349 - val_acc: 0.9895
Epoch 8/150 - loss: 0.0214 - acc: 0.9950 - val_loss: 0.0467 - val_acc: 0.9855
Epoch 9/150 - loss: 0.0295 - acc: 0.9910 - val_loss: 0.0566 - val_acc: 0.9855
Epoch 10/150 - loss: 0.0111 - acc: 0.9974 - val_loss: 0.0286 - val_acc: 0.9918
Epoch 11/150 - loss: 0.0054 - acc: 0.9985 - val_loss: 0.0279 - val_acc: 0.9927
Epoch 12/150 - loss: 0.0009 - acc: 1.0000 - val_loss: 0.0239 - val_acc: 0.9936
Epoch 13/150 - loss: 0.0004 - acc: 

In [15]:
# Сохранение моделей
import pickle

# Собственная модель
with open('model_my.pkl', 'wb') as f:
    pickle.dump({'weights': my_net.weights, 'biases': my_net.biases, 'sizes': my_net.sizes}, f)
print("Моя модель сохранена в model_my.pkl")

# # Keras
# keras_model.save('model_keras.h5')
# print("Keras сохранена в model_keras.h5")

# PyTorch
torch.save(pt_model.state_dict(), 'model_pytorch.pth')
print("PyTorch сохранена в model_pytorch.pth")

Моя модель сохранена в model_my.pkl
PyTorch сохранена в model_pytorch.pth


---
## Заключение

В ходе выполнения лабораторной работы были изучены и реализованы три различных подхода к созданию и обучению нейронных сетей для задачи классификации рукописных символов.

### Основные результаты:

1. **Собственная реализация на NumPy**
   - Реализован полный цикл обучения нейронной сети: прямой проход, вычисление функции потерь, обратное распространение ошибки, обновление весов
   - Использована функция активации `tanh(3x)` с коэффициентом α=3 для ускорения сходимости
   - Применены: инициализация весов He, SGD с мини-батчами, early stopping
   - **Точность: 98.91%**

2. **Реализация на Keras (TensorFlow)**
   - Показана высокая скорость разработки — модель создаётся в 5-10 раз быстрее
   - Встроенные механизмы: автоматическое дифференцирование, оптимизаторы, callback-функции
   - **Проведён подбор alpha: лучший = 2**
   - **Точность с alpha=2: 99.27%**

3. **Реализация на PyTorch**
   - Demonstrated гибкость управления процессом обучения
   - Прозрачная отладка благодаря динамическому графу вычислений
   - **Проведён подбор alpha: лучший = 1**
   - **Точность с alpha=1: 99.41%**

### Сравнение лучших alpha:

| Фреймворк | Лучший alpha | Точность | Потери |
|-----------|--------------|----------|--------|
| Keras     | 2            | 99.27%   | 0.0272 |
| PyTorch   | 1            | 99.41%   | 0.0217 |

### Выводы по подбору alpha:
- **Keras**: оптимальное значение alpha=2. При больших значениях (4-5) точность падает из-за слишком крутой функции активации.
- **PyTorch**: оптимальное значение alpha=1. Более пологая функция активации обеспечивает лучшую сходимость.
- При увеличении alpha > 3 наблюдается ухудшение обобщающей способности (переобучение).

---
**Выполнил:** Привалихин Дмитрий Сергеевич, ИСИб-23-1
